# Calculating Naive Bayes With Features

In this first demo, we will build a Naive Bayes classifier against the iris data set. We will use scikit-learn's `GaussianNB` to train a model against a subset of the data and then test it against a holdout set.

We'll use scikit-learn's `confusion_matrix` and `classification_report` to evaluate our model.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

First, load the iris data set.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame
df.head()

We will take a slice of records out and hold it as a test data set. We avoid using it for training the model. That way, we get a realistic view of how the model behaves on unseen data points.

We use `train_test_split` to shuffle and split the data. The `stratify` parameter ensures a representative slice of the three species of iris.

We're saving 20% of the data for testing. You can adjust this as needed.

In [ ]:
X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1773, stratify=y
)

Generating a Naive Bayes classifier is a one-liner once the data is set up.

In [ ]:
nb = GaussianNB()
nb.fit(X_train, y_train)

Let's visualize the distribution of each feature by species. This helps us eyeball which variables are more important for discerning species.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, (feature, ax) in enumerate(zip(iris.feature_names, axes.ravel())):
    for species_id, species_name in enumerate(iris.target_names):
        subset = X_train[y_train == species_id]
        ax.hist(subset[feature], alpha=0.5, label=species_name, bins=15)
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()

Once we have a trained model, let's run `predict` against our test data set and combine the output with our test data.

In [ ]:
predictions = nb.predict(X_test)

iris_output = X_test.copy()
iris_output["species"] = y_test
iris_output["prediction"] = predictions
iris_output.head()

Now we can generate a confusion matrix. scikit-learn's `classification_report` provides precision, recall, and F1-score. These map to the concepts of positive/negative predictive value and sensitivity/specificity.

**Accuracy** is the simplest measure: of all predictions, how many were correct? It is defined as (correct predictions) / (total predictions). Accuracy can mislead when one class is much more common than another.

**Positive predictive value (Precision)** for a category: if the model predicts that inputs match a particular class, what is the probability that this judgement is correct? It is defined as (true positives) / (true positives + false positives).

**Negative predictive value** for a category: if the model predicts that inputs do *not* match a particular class, what is the probability that this judgement is correct? It is defined as (true negatives) / (true negatives + false negatives).

**Recall (Sensitivity)** for a category: of all items that truly belong to a particular class, how many did the model correctly identify? It is defined as (true positives) / (true positives + false negatives).

In [ ]:
print(classification_report(y_test, predictions, target_names=iris.target_names))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, predictions, display_labels=iris.target_names, cmap="Blues"
)
plt.title("Naive Bayes — Iris Classification")
plt.show()